# Thesis figures

Every figure in the report, generated from the artifacts on disk. One function per
figure — run a cell, get the figure; `save(fig, "name")` writes SVG + PNG into
`figures/`.

**This notebook replaces `EDA_final.py`** (958 lines of classify-era analysis, deleted
2026-08-20 along with the dataset it read). Nothing here depends on the retired 4-class
task.

**What it reads** — all current, all on disk:

| source | feeds |
|---|---|
| `data/grid_dataset_<tag>_n1{,_meta}.json{l}` | §1 the data |
| `results/failures/failures_<tag>.jsonl` | §2 how the model fails |
| `rules_35b/`, `translated_rules/`, `validated_{strict,translated}/` | §3 extraction |
| `kg/knowledge_graph.json` | §4 rules and provenance |
| `results/shield/shield_<tag>_{validated,run3}.json` | §5 the shield |

Run `Kernel → Restart & Run All` to regenerate everything.


In [ ]:
from __future__ import annotations

import json
import glob
import os
import re
from collections import Counter
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

FIG_DIR = Path("figures")
TAGS = ["neurips2020", "case14", "wcci2022"]
NICE = {"neurips2020": "36-bus\n(trained on)", "case14": "14-bus\n(unseen)", "wcci2022": "118-bus\n(unseen)"}
C_MODEL, C_RULE, C_SHIELD, C_MUTED = "#2f4b7c", "#ff7c43", "#d45087", "#b4b2a9"

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight", "font.size": 9,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 10.5, "axes.labelsize": 9, "legend.frameon": False,
})


def save(fig, name, formats=("svg", "png")):
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    out = []
    for f in formats:
        p = FIG_DIR / f"{name}.{f}"
        fig.savefig(p)
        out.append(str(p))
    return out


def _jsonl(path, limit=None):
    n = 0
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)
            n += 1
            if limit and n >= limit:
                return


def load_shield(tag, corpus="validated"):
    return json.load(open(f"results/shield/shield_{tag}_{corpus}.json", encoding="utf-8"))


def _bar_labels(ax, bars, fmt="{:.3f}", dy=0.0, fs=8):
    for b in bars:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + dy,
                fmt.format(b.get_height()), ha="center", va="bottom", fontsize=fs)

## 1. The data — what the model is asked to predict


**`fig_topology_scale`** — Scale of the three grids. The model trains on the smallest and is tested on two it never sees.


In [ ]:
def fig_topology_scale():
    """Scale of the three grids and how much N-1 work each implies."""
    meta = {t: json.load(open(f"data/grid_dataset_{t}_n1_meta.json", encoding="utf-8")) for t in TAGS}
    sh = {t: load_shield(t) for t in TAGS}
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.1))
    for ax, (key, label, src) in zip(axes, [
        ("n_sub", "substations", lambda t: meta[t]["n_sub"]),
        ("n_line", "transmission lines", lambda t: meta[t]["n_line"]),
        ("n_cont", "contingencies scored", lambda t: sh[t]["n_contingencies"]),
    ]):
        vals = [src(t) for t in TAGS]
        bars = ax.bar([NICE[t] for t in TAGS], vals,
                      color=[C_MODEL, C_MUTED, C_MUTED], width=0.62)
        _bar_labels(ax, bars, "{:,.0f}", dy=max(vals) * 0.02)
        ax.set_title(label)
        ax.margins(y=0.18)
        ax.tick_params(labelsize=8)
    axes[0].set_ylabel("count")
    fig.suptitle("The model is trained on the smallest grid and tested on two it has never seen",
                 fontsize=10.5, y=1.04)
    fig.tight_layout()
    return fig


_out = fig_topology_scale()
_fig = _out[0] if isinstance(_out, tuple) else _out
save(_fig, "topology_scale")
_fig

**`fig_mixed_frames`** — The measurement that rules out a graph-level model. **Note the correction:** no frame anywhere is entirely secure (0.00% on all three grids), but 96–99% are *strictly* mixed — not 100%, as earlier drafts said.


In [ ]:
def fig_mixed_frames(n_frames=None):
    """Within one grid state, some contingencies violate and others do not.

    This is the measurement that rules out a graph-level model: if frames were
    all-or-nothing, one label per frame would do.

    ⚠ Two different statistics live here and the docs previously conflated them.
    Measured over EVERY frame of all three datasets (22,000 frames):
      - frames with zero violations : 0.00% on all three grids — no frame is
        entirely secure, which is the "100%" figure the older docs quoted;
      - strictly mixed frames       : 96.4-98.8%, NOT 100%. The remainder are
        frames where every contingency violates.
    """
    fig, ax = plt.subplots(figsize=(7.6, 3.5))
    rows = []
    for t, colour in zip(TAGS, [C_MODEL, C_RULE, C_SHIELD]):
        fracs = []
        for rec in _jsonl(f"data/grid_dataset_{t}_n1.jsonl", limit=n_frames):
            v = np.asarray(rec.get("n1_violation", []), dtype=float)
            v = v[v >= 0]
            if v.size:
                fracs.append(v.mean())
        fracs = np.asarray(fracs)
        mixed = float(((fracs > 0) & (fracs < 1)).mean())
        clean = float((fracs == 0).mean())
        rows.append((t, len(fracs), mixed, clean))
        ax.hist(fracs, bins=np.linspace(0, 1, 41), alpha=0.55, color=colour,
                label=f"{NICE[t].replace(chr(10), ' ')} — {mixed:.1%} mixed")
    ax.set_xlabel("fraction of this frame's contingencies that violate a limit")
    ax.set_ylabel("frames")
    ax.set_title("Every frame contains both safe and unsafe contingencies\n"
                 "(no frame anywhere is entirely secure; 96-99% are strictly mixed)",
                 fontsize=9.8)
    ax.legend(fontsize=8)
    fig.tight_layout()
    return fig, rows


_out = fig_mixed_frames()
_fig = _out[0] if isinstance(_out, tuple) else _out
save(_fig, "mixed_frames")
_fig

**`fig_rho_threshold_ceiling`** — The model against the best single-threshold rule. It wins at home and loses outright on the 14-bus grid.


In [ ]:
def fig_rho_threshold_ceiling():
    """Why a single rho threshold cannot solve this: the model's own baseline."""
    sh = {t: load_shield(t) for t in TAGS}
    fig, ax = plt.subplots(figsize=(7.2, 3.4))
    x = np.arange(len(TAGS))
    w = 0.36
    rule = [sh[t]["f1_rule_baseline"] for t in TAGS]
    model = [sh[t]["f1_model_best_on_topology"] for t in TAGS]
    b1 = ax.bar(x - w / 2, rule, w, label="best single rule on removed-line rho", color=C_RULE)
    b2 = ax.bar(x + w / 2, model, w, label="the trained model", color=C_MODEL)
    _bar_labels(ax, b1, dy=0.012)
    _bar_labels(ax, b2, dy=0.012)
    for i, t in enumerate(TAGS):
        ratio = model[i] / rule[i]
        ax.text(i, max(rule[i], model[i]) + 0.09, f"{ratio:.2f}x",
                ha="center", fontsize=9,
                color=("#993c1d" if ratio < 1 else "#0f6e56"), fontweight="bold")
    ax.set_xticks(x, [NICE[t] for t in TAGS], fontsize=8)
    ax.set_ylabel("F1")
    ax.set_ylim(0, 1.05)
    ax.set_title("The model beats the threshold baseline at home, and loses to it on the small grid")
    ax.legend(fontsize=8, loc="upper right")
    fig.tight_layout()
    return fig


_out = fig_rho_threshold_ceiling()
_fig = _out[0] if isinstance(_out, tuple) else _out
save(_fig, "rho_threshold_ceiling")
_fig

## 2. The model — how it fails


**`fig_failure_modes`** — What kind of mistake the model makes on each grid, from the per-contingency failure log.


In [ ]:
def fig_failure_modes(max_records=200_000):
    """What kind of mistake the model makes, per grid."""
    order = ["missed_violation_uncaught", "missed_violation_caught", "false_alarm", "correct"]
    pretty = {"missed_violation_uncaught": "missed violation\n(shield could not see it)",
              "missed_violation_caught": "missed violation\n(shield caught it)",
              "false_alarm": "false alarm", "correct": "correct"}
    counts = {}
    for t in TAGS:
        c = Counter()
        for rec in _jsonl(f"results/failures/failures_{t}.jsonl", limit=max_records):
            c[rec.get("mode", "?")] += 1
        counts[t] = c
    modes = [m for m in order if any(counts[t].get(m) for t in TAGS)]
    modes += sorted({m for t in TAGS for m in counts[t]} - set(modes))

    fig, ax = plt.subplots(figsize=(7.6, 3.4))
    bottoms = np.zeros(len(TAGS))
    palette = [C_SHIELD, "#0f6e56", C_RULE, C_MUTED, "#7f77dd"]
    for i, m in enumerate(modes):
        vals = np.array([counts[t].get(m, 0) for t in TAGS], dtype=float)
        ax.barh([NICE[t].replace("\n", " ") for t in TAGS], vals, left=bottoms,
                color=palette[i % len(palette)], label=pretty.get(m, m), height=0.55)
        bottoms += vals
    ax.set_xlabel("logged records (capped at 50,000 per grid by the harness)")
    ax.set_title("Failure composition — what the shield is being asked to catch")
    ax.legend(fontsize=7.5, ncol=2, loc="lower right")
    ax.invert_yaxis()
    fig.tight_layout()
    return fig, counts


_out = fig_failure_modes()
_fig = _out[0] if isinstance(_out, tuple) else _out
save(_fig, "failure_modes")
_fig

## 3. Extraction — 2,463 rules become 4


**`fig_extraction_funnel`** — The headline number for Component B. Every stage is a measured count, not an estimate.


In [ ]:
def _funnel_counts():
    n_cand = sum(1 for p in glob.glob("rules_35b/*_candidates.jsonl") for _ in _jsonl(p))
    trans = json.load(open("translated_rules/translation_run_summary.json", encoding="utf-8"))
    guard = json.load(open("translated_rules/guarded/polarity_guard_summary.json", encoding="utf-8"))
    val = json.load(open("validated_translated/validation_run_summary.json", encoding="utf-8"))
    n_dedup = sum(1 for _ in _jsonl("validated_translated/all_rules_deduped.jsonl"))
    return [
        ("stage 1 — extracted from 16 PDFs", n_cand),
        ("stage 2 — expressible in simulator terms", trans["total_translatable"]),
        ("stage 2.5 — survives the polarity guard", guard["n_kept"]),
        ("stage 3 — confirmed against source text", val["total_confirmed"]),
        ("deduplicated — what the shield is served", n_dedup),
    ]

def fig_extraction_funnel():
    """2,463 candidate rules become 4. The headline number for Component B."""
    rows = _funnel_counts()
    labels = [r[0] for r in rows]
    vals = [r[1] for r in rows]
    fig, ax = plt.subplots(figsize=(8.2, 3.6))
    y = np.arange(len(rows))[::-1]
    bars = ax.barh(y, vals, color=[C_MODEL] + [C_MUTED] * 3 + [C_SHIELD], height=0.6)
    ax.set_yticks(y, labels, fontsize=8.5)
    ax.set_xscale("log")
    ax.set_xlabel("rules (log scale)")
    for b, v, prev in zip(bars, vals, [None] + vals[:-1]):
        txt = f"{v:,}" + (f"   ({v / vals[0]:.2%} of intake)" if prev is not None else "")
        ax.text(v * 1.12, b.get_y() + b.get_height() / 2, txt, va="center", fontsize=8.5)
    ax.set_xlim(0.7, vals[0] * 12)
    ax.set_title("The extraction funnel — 2,463 candidates yield 4 usable rules (0.16%)")
    fig.tight_layout()
    return fig, rows


_out = fig_extraction_funnel()
_fig = _out[0] if isinstance(_out, tuple) else _out
save(_fig, "extraction_funnel")
_fig

**`fig_per_document_yield`** — The standards/simulator mismatch is not uniform. A dot plot, not stacked bars — on a log axis bar *length* is not proportional to value, so bars would misrepresent this.


In [ ]:
def fig_per_document_yield():
    """The standards/simulator mismatch is not uniform across documents."""
    cand = {Path(p).name.replace("_candidates.jsonl", ""): sum(1 for _ in _jsonl(p))
            for p in glob.glob("rules_35b/*_candidates.jsonl")}
    trans = {Path(p).name.replace("_translated.jsonl", ""): sum(1 for _ in _jsonl(p))
             for p in glob.glob("translated_rules/*_translated.jsonl")}
    conf = {Path(p).name.replace("_confirmed.jsonl", ""): sum(1 for _ in _jsonl(p))
            for p in glob.glob("validated_translated/*_confirmed.jsonl")}
    docs = sorted(cand, key=lambda d: -cand[d])
    short = [d.replace("_", " ")[:34] + ("…" if len(d) > 34 else "") for d in docs]

    # Dot plot, not stacked bars. On a log axis a bar's LENGTH is not proportional
    # to its value, so overlaid bars would make "7 of 571 expressible" look like a
    # near-miss. Position encodes the value honestly; the connecting line shows the
    # drop.
    fig, ax = plt.subplots(figsize=(8.8, 0.34 * len(docs) + 1.8))
    y = np.arange(len(docs))
    for i, d in enumerate(docs):
        c, t_, v = cand[d], trans.get(d, 0), conf.get(d, 0)
        ax.plot([max(v, 0.5), c], [i, i], color="#d3d1c7", lw=1.4, zorder=1, solid_capstyle="round")
        ax.scatter([c], [i], s=34, color=C_MUTED, zorder=3, label="extracted" if i == 0 else None)
        if t_:
            ax.scatter([t_], [i], s=34, color=C_MODEL, zorder=4, label="expressible" if i == 0 else None)
        if v:
            ax.scatter([v], [i], s=44, color=C_SHIELD, zorder=5, marker="D",
                       label="validated" if i == 0 else None)
        ax.text(c * 1.25, i, f"{c:,} → {t_} → {v}", va="center", fontsize=7.5, color="#5f5e5a")
    ax.set_yticks(y, short, fontsize=7.5)
    ax.set_xscale("log")
    ax.set_xlim(0.4, max(cand.values()) * 6)
    ax.set_xlabel("rules surviving each stage (log scale — position, not bar length, encodes the value)")
    ax.invert_yaxis()
    ax.grid(axis="x", ls=":", lw=0.5, color="#d3d1c7")
    ax.set_axisbelow(True)
    ax.legend(fontsize=8, loc="lower right")
    ax.set_title("Per-document yield — the mismatch is not uniform.\n"
                 "TPL-001-5.1 and NERC FAC-008-5, the two documents most about post-contingency\n"
                 "performance, produced 15 candidates between them.", fontsize=9.5)
    fig.tight_layout()
    return fig


_out = fig_per_document_yield()
_fig = _out[0] if isinstance(_out, tuple) else _out
save(_fig, "per_document_yield")
_fig

**`fig_untranslatable_reasons`** — Why 2,405 candidates could not be expressed. The buckets come from the translator's own reason prefixes, with a keyword fallback for the unprefixed ones.


In [ ]:
def _reason_bucket(reason):
    r = (reason or "").strip().lower()
    head = r.split(":")[0].strip() if ":" in r[:40] else ""
    named = {
        "frequency": "frequency (not simulated at all)",
        "ride-through": "ride-through band (needs duration)",
        "compound": "compound / mixed requirement",
        "unobservable": "quantity the simulator lacks",
        "unmeasurable": "quantity the simulator lacks",
        "scope": "out of scope for the observation",
        "rocof": "frequency (not simulated at all)",
        "droop": "control-loop behaviour",
        "time_seconds": "needs a time / duration term",
        "translator_fail": "translator failure",
    }
    if head in named:
        return named[head]
    for pat, bucket in [
        (r"frequen|hz|rocof", "frequency (not simulated at all)"),
        (r"ride[- ]through|duration|time_sec|seconds|minute", "needs a time / duration term"),
        (r"protect|relay|breaker|trip setting|scheme", "protection & control settings"),
        (r"notif|report|document|procedur|complian|agreement", "administrative / regulatory"),
        (r"capab|nameplate|manufactur|design|equipment|rating plate", "equipment nameplate data"),
    ]:
        if re.search(pat, r):
            return bucket
    return "quantity the simulator lacks"

def fig_untranslatable_reasons():
    """Why 2,405 of 2,463 rules could not be expressed — the mismatch, itemised."""
    c = Counter()
    for p in glob.glob("translated_rules/*_untranslatable.jsonl"):
        for rec in _jsonl(p):
            c[_reason_bucket(rec.get("reason"))] += 1
    items = c.most_common()
    total = sum(c.values())
    fig, ax = plt.subplots(figsize=(7.8, 0.42 * len(items) + 1.5))
    y = np.arange(len(items))[::-1]
    bars = ax.barh(y, [v for _, v in items], color=C_MUTED, height=0.6)
    bars[0].set_color(C_MODEL)
    ax.set_yticks(y, [k for k, _ in items], fontsize=8.5)
    for b, (_, v) in zip(bars, items):
        ax.text(v + total * 0.008, b.get_y() + b.get_height() / 2,
                f"{v:,}  ({v/total:.0%})", va="center", fontsize=8)
    ax.set_xlabel(f"rules ({total:,} untranslatable of 2,463)")
    ax.margins(x=0.16)
    ax.set_title("Why the corpus is thin — the standards regulate what the simulator does not model")
    fig.tight_layout()
    return fig, items


_out = fig_untranslatable_reasons()
_fig = _out[0] if isinstance(_out, tuple) else _out
save(_fig, "untranslatable_reasons")
_fig

**`fig_validation_ab`** — The controlled A/B: same model, same 32 rules, one changed question in the prompt.


In [ ]:
def fig_validation_ab():
    """The A/B that showed the validator's question, not the corpus, was the problem."""
    arms = {}
    for arm in ("strict", "translated"):
        s = json.load(open(f"validated_{arm}/validation_run_summary.json", encoding="utf-8"))
        arms[arm] = (s["total_confirmed"], s["total_rejected"], s["n_unique_rules_after_dedup"])
    fig, ax = plt.subplots(figsize=(7.2, 3.2))
    x = np.arange(2)
    w = 0.36
    conf = [arms[a][0] for a in ("strict", "translated")]
    rej = [arms[a][1] for a in ("strict", "translated")]
    b1 = ax.bar(x - w / 2, conf, w, label="confirmed", color=C_SHIELD)
    b2 = ax.bar(x + w / 2, rej, w, label="rejected", color=C_MUTED)
    _bar_labels(ax, b1, "{:.0f}", dy=0.4)
    _bar_labels(ax, b2, "{:.0f}", dy=0.4)
    ax.set_xticks(x, ["strict\n\"is it stated in the text?\"",
                      "translated\n\"is it a faithful operationalization?\""], fontsize=8.5)
    ax.set_ylabel("rules (of 32 guarded)")
    ax.set_title("Same model, same input, one changed question — 1 confirmed becomes 11")
    ax.legend(fontsize=8)
    ax.margins(y=0.18)
    fig.tight_layout()
    return fig, arms


_out = fig_validation_ab()
_fig = _out[0] if isinstance(_out, tuple) else _out
save(_fig, "validation_ab")
_fig

## 4. The rules and where they came from


**`fig_vocabulary_coverage`** — How much of the 14-variable observable vocabulary the corpus actually reaches — and that it says nothing about topology.


In [ ]:
def fig_vocabulary_coverage():
    """How much of the 14-variable vocabulary the corpus actually reaches."""
    import sys
    sys.path.append(".")
    from extraction.common import CONDITION_VOCABULARY
    from kg.build import used_variables

    stages = {}
    stages["translated (58)"] = [r["rule"]["condition"] for p in glob.glob("translated_rules/*_translated.jsonl")
                                 for r in _jsonl(p)]
    stages["guarded (32)"] = [r["rule"]["condition"] for p in glob.glob("translated_rules/guarded/*_translated.jsonl")
                              for r in _jsonl(p)]
    stages["validated (4)"] = [r["condition"] for r in _jsonl("validated_translated/all_rules_deduped.jsonl")]

    vocab = list(CONDITION_VOCABULARY)
    grid = np.zeros((len(stages), len(vocab)))
    for i, conds in enumerate(stages.values()):
        used = Counter(v for c in conds for v in used_variables(c))
        for j, name in enumerate(vocab):
            grid[i, j] = used.get(name, 0)

    fig, ax = plt.subplots(figsize=(9.2, 2.6))
    ax.imshow(np.where(grid > 0, 1, 0), aspect="auto", cmap="Blues", vmin=0, vmax=1.6)
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            if grid[i, j]:
                ax.text(j, i, int(grid[i, j]), ha="center", va="center", fontsize=7.5, color="#042c53")
    ax.set_xticks(range(len(vocab)), vocab, rotation=42, ha="right", fontsize=7.5)
    ax.set_yticks(range(len(stages)), list(stages), fontsize=8.5)
    reached = int((grid.sum(axis=0) > 0).sum())
    ax.set_title(f"Vocabulary coverage — the corpus reaches {reached} of {len(vocab)} observable "
                 f"variables, and nothing about topology")
    fig.tight_layout()
    return fig, grid


_out = fig_vocabulary_coverage()
_fig = _out[0] if isinstance(_out, tuple) else _out
save(_fig, "vocabulary_coverage")
_fig

## 5. The shield — what the gate is worth


**`fig_shield_effect`** — The headline result, per grid.


In [ ]:
def fig_shield_effect():
    """The headline: what the gate is worth, per grid."""
    sh = {t: load_shield(t) for t in TAGS}
    fig, ax = plt.subplots(figsize=(7.4, 3.4))
    x = np.arange(len(TAGS))
    w = 0.36
    g = [sh[t]["arms"]["gnn"]["f1"] for t in TAGS]
    s = [sh[t]["arms"]["gnn_shield"]["f1"] for t in TAGS]
    b1 = ax.bar(x - w / 2, g, w, label="model alone", color=C_MODEL)
    b2 = ax.bar(x + w / 2, s, w, label="model + symbolic gate", color=C_SHIELD)
    _bar_labels(ax, b1, dy=0.012)
    _bar_labels(ax, b2, dy=0.012)
    for i in range(len(TAGS)):
        ax.text(i, max(g[i], s[i]) + 0.075, f"+{s[i]-g[i]:.4f}", ha="center",
                fontsize=9, fontweight="bold", color="#0f6e56")
    ax.set_xticks(x, [NICE[t] for t in TAGS], fontsize=8)
    ax.set_ylabel("F1")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8, loc="upper right")
    ax.set_title("The gate is worth most where the model is weakest")
    fig.tight_layout()
    return fig


_out = fig_shield_effect()
_fig = _out[0] if isinstance(_out, tuple) else _out
save(_fig, "shield_effect")
_fig

**`fig_intervention_precision`** — The finding: the rulebook's accuracy is topology-invariant. Only its reach changes.


In [ ]:
def fig_intervention_precision():
    """The finding: the gate's accuracy is flat across topologies; its reach is not."""
    sh = {t: load_shield(t) for t in TAGS}
    prec = [sh[t]["corrections"] / sh[t]["blocked"] for t in TAGS]
    reach = [sh[t]["eligible"] / sh[t]["n_contingencies"] for t in TAGS]
    ceil = [sh[t]["missed_reachable_by_rule"] / sh[t]["missed_violations"] for t in TAGS]

    fig, axes = plt.subplots(1, 3, figsize=(10.2, 3.2))
    for ax, vals, title, ylab, fmt, colour in [
        (axes[0], prec, "when it acts, how often it is right", "intervention precision", "{:.1%}", C_SHIELD),
        (axes[1], reach, "how often it gets to act", "share of contingencies", "{:.2%}", C_MODEL),
        (axes[2], ceil, "share of model error it can reach", "of missed violations", "{:.1%}", C_RULE),
    ]:
        bars = ax.bar([NICE[t] for t in TAGS], vals, color=colour, width=0.62)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v + max(vals) * 0.03, fmt.format(v),
                    ha="center", fontsize=8.5)
        ax.set_title(title, fontsize=9.5)
        ax.set_ylabel(ylab, fontsize=8.5)
        ax.margins(y=0.22)
        ax.tick_params(labelsize=8)
    axes[0].axhline(np.mean(prec), ls="--", lw=0.9, color="#5f5e5a")
    axes[0].text(2.45, np.mean(prec), f" mean {np.mean(prec):.1%}", fontsize=7.5,
                 va="center", color="#5f5e5a")
    fig.suptitle("The rulebook's accuracy is topology-invariant. Only its reach changes.",
                 fontsize=10.5, y=1.05)
    fig.tight_layout()
    return fig, dict(zip(TAGS, zip(prec, reach, ceil)))


_out = fig_intervention_precision()
_fig = _out[0] if isinstance(_out, tuple) else _out
save(_fig, "intervention_precision")
_fig

**`fig_corpus_comparison`** — Validation removed 28 of 32 rules and every metric improved or held.


In [ ]:
def fig_corpus_comparison():
    """Validation removed 28 of 32 rules and every metric improved or held."""
    fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.3))
    x = np.arange(len(TAGS))
    w = 0.36
    for ax, key, title, fmt in [
        (axes[0], "delta", "F1 gain from the gate", "{:+.4f}"),
        (axes[1], "prec", "intervention precision", "{:.1%}"),
    ]:
        for k, (corpus, colour, off) in enumerate([("run3", C_MUTED, -w / 2), ("validated", C_SHIELD, w / 2)]):
            vals = []
            for t in TAGS:
                d = load_shield(t, corpus)
                vals.append(d["arms"]["gnn_shield"]["f1"] - d["arms"]["gnn"]["f1"] if key == "delta"
                            else d["corrections"] / d["blocked"])
            bars = ax.bar(x + off, vals, w, color=colour,
                          label="guarded corpus (32 rules)" if corpus == "run3" else "validated corpus (4 rules)")
            for b, v in zip(bars, vals):
                ax.text(b.get_x() + b.get_width() / 2, v + max(vals) * 0.03, fmt.format(v),
                        ha="center", fontsize=7.5)
        ax.set_xticks(x, [NICE[t] for t in TAGS], fontsize=8)
        ax.set_title(title, fontsize=9.5)
        ax.margins(y=0.22)
    axes[0].legend(fontsize=7.5, loc="upper left")
    fig.suptitle("Dropping 28 of 32 rules improved or held every metric on every grid",
                 fontsize=10.5, y=1.04)
    fig.tight_layout()
    return fig


_out = fig_corpus_comparison()
_fig = _out[0] if isinstance(_out, tuple) else _out
save(_fig, "corpus_comparison")
_fig

## The knowledge-graph figures

These two are produced by `kg/figures.py` and written by `scripts/build_kg.py --figures`.
They are re-exposed here so one notebook regenerates every figure in the report.


In [ ]:
def fig_kg_provenance():
    from kg.build import load_kg
    from kg.figures import plot_provenance
    return plot_provenance(load_kg("kg/knowledge_graph.json"), FIG_DIR / "kg_provenance")


def fig_kg_corroboration():
    from kg.build import load_kg
    from kg.figures import plot_corroboration
    return plot_corroboration(load_kg("kg/knowledge_graph.json"), FIG_DIR / "kg_corroboration")


print(fig_kg_provenance())
print(fig_kg_corroboration())

## Regenerate everything

One call, for when the underlying artifacts change.


In [ ]:
ALL_FIGURES = [
    "topology_scale", "mixed_frames", "rho_threshold_ceiling",
    "failure_modes",
    "extraction_funnel", "per_document_yield", "untranslatable_reasons", "validation_ab",
    "vocabulary_coverage",
    "shield_effect", "intervention_precision", "corpus_comparison",
]


def regenerate_all(formats=("svg", "png")):
    """Rebuild every figure into figures/. Returns {name: [paths]}."""
    written = {}
    for name in ALL_FIGURES:
        out = globals()[f"fig_{name}"]()
        fig = out[0] if isinstance(out, tuple) else out
        written[name] = save(fig, name, formats)
        plt.close(fig)
    written["kg_provenance"] = fig_kg_provenance()
    written["kg_corroboration"] = fig_kg_corroboration()
    return written


for _name, _paths in regenerate_all().items():
    print(f"{_name:<24} {len(_paths)} file(s)")